In [2]:
import numpy as np, torch

# Neuromancer imports: modeling, loss, logging, system dynamics
import neuromancer.psl as psl
from neuromancer.system import Node, System
from neuromancer.dynamics import ode, integrators
from neuromancer.plot import pltCL, pltPhase

# SparseDPC-specific modules: libraries, data, training, utils
from SparseDPC.sindy.fx_library import fx_library , fx_policy_library
from SparseDPC.sindy.fx_library import fx_policy_library
from utils import *

# Set random seed for reproducibility
torch.manual_seed(0)

# Use GPU if available, otherwise fallback to CPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


In [6]:
# Ground truth system model
from utils import DoubleIntegrator2D


gt_model = DoubleIntegrator2D()
gt_model.params
# Sampling rate
ts = gt_model.params[1]["ts"]

# Problem dimensions
nx = gt_model.nx      # number of states
nu = gt_model.nu      # number of control inputs
nref = nx             # reference dimensions equal to state dimensions

# Constraint bounds
umin = -2.
umax = +2.
xmin = -5.
xmax = +5.


# Create one fx model per state dimension
# Create one fx model per state dimension
fx_models = [fx_library(i, nx=nx, nu=nu, seed=0, device=device,
                           max_degree=2, add_sqrt=False, add_u_prod=False) for i in range(nx)]
# Print model structures
for fx in fx_models:
    print(fx)


import os, glob, torch

run_id = 1
models_dir = f"results/tests/dynamics/run_{run_id}/saved_models"   # change root here

for i, model in enumerate(fx_models):
    pattern = os.path.join(models_dir, f"trained_dynamics_x{i}_*.pth")
    matches = glob.glob(pattern)
    if not matches:
        raise FileNotFoundError(f"No checkpoint for x{i} in {models_dir}")

    ckpt = max(matches, key=os.path.getmtime)
    state_dict = torch.load(ckpt, weights_only=True)
    model.load_state_dict(state_dict)
    model.to(device).eval()
    print(f"Loaded x{i} from {os.path.basename(ckpt)}")


dx0/dt = -0.000*x0 + 0.027*x1 + -0.041*x2 + -0.037*x3 + -0.019*x0^2 + 0.013*x0*x1 + -0.001*x0*x2 + 0.040*x0*x3 + -0.004*x1^2 + 0.013*x1*x2 + -0.015*x1*x3 + -0.010*x2^2 + -0.048*x2*x3 + -0.033*x3^2 + -0.021*u_0 + 0.002*u_1 + 0.020*x0*u_0 + 0.030*x1*u_0 + -0.034*x2*u_0 + -0.022*x3*u_0 + 0.018*x0*u_1 + 0.042*x1*u_1 + -0.010*x2*u_1 + 0.037*x3*u_1

dx1/dt = -0.000*x1 + 0.027*x0 + -0.041*x2 + -0.037*x3 + -0.019*x0^2 + 0.013*x0*x1 + -0.001*x0*x2 + 0.040*x0*x3 + -0.004*x1^2 + 0.013*x1*x2 + -0.015*x1*x3 + -0.010*x2^2 + -0.048*x2*x3 + -0.033*x3^2 + -0.021*u_0 + 0.002*u_1 + 0.020*x0*u_0 + 0.030*x1*u_0 + -0.034*x2*u_0 + -0.022*x3*u_0 + 0.018*x0*u_1 + 0.042*x1*u_1 + -0.010*x2*u_1 + 0.037*x3*u_1

dx2/dt = -0.000*x2 + 0.027*x0 + -0.041*x1 + -0.037*x3 + -0.019*x0^2 + 0.013*x0*x1 + -0.001*x0*x2 + 0.040*x0*x3 + -0.004*x1^2 + 0.013*x1*x2 + -0.015*x1*x3 + -0.010*x2^2 + -0.048*x2*x3 + -0.033*x3^2 + -0.021*u_0 + 0.002*u_1 + 0.020*x0*u_0 + 0.030*x1*u_0 + -0.034*x2*u_0 + -0.022*x3*u_0 + 0.018*x0*u_1 + 0.042*x